In [ ]:
import os
import pandas as pd

files = [
    "nav_history_cleaned.csv",
    "investor_transactions_cleaned.csv",
    "scheme_performance_cleaned.csv",
    "cleaned_01_fund_master.csv",
    "cleaned_03_aum_by_fund_house.csv",
    "cleaned_04_monthly_sip_inflows.csv",
    "cleaned_05_category_inflows.csv",
    "cleaned_06_industry_folio_count.csv",
    "cleaned_09_portfolio_holdings.csv",
    "cleaned_10_benchmark_indices.csv"
]

for f in files:
    if os.path.exists(f):
        df = pd.read_csv(f, nrows=2)
        print(f"{f}: columns={list(df.columns)}")
    else:
        print(f"{f} does not exist")

nav_history_cleaned.csv: columns=['amfi_code', 'date', 'nav']
investor_transactions_cleaned.csv: columns=['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']
scheme_performance_cleaned.csv: columns=['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']
cleaned_01_fund_master.csv: columns=['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']
cleaned_03_aum_by_fund_house.csv: columns=['date', 'fund_house', 'aum_lakh_crore', 'aum_crore'

In [ ]:
import sqlite3
import pandas as pd
import glob


conn = sqlite3.connect('mutual_funds.db')

csv_files = {
    'fund_master': 'cleaned_01_fund_master.csv',
    'aum_by_fund_house': 'cleaned_03_aum_by_fund_house.csv',
    'monthly_sip_inflows': 'cleaned_04_monthly_sip_inflows.csv',
    'category_inflows': 'cleaned_05_category_inflows.csv',
    'industry_folio_count': 'cleaned_06_industry_folio_count.csv',
    'portfolio_holdings': 'cleaned_09_portfolio_holdings.csv',
    'benchmark_indices': 'cleaned_10_benchmark_indices.csv',
    'nav_history': 'nav_history_cleaned.csv',
    'investor_transactions': 'investor_transactions_cleaned.csv',
    'scheme_performance': 'scheme_performance_cleaned.csv'
}

print("Loading tables into SQLite database...")
for table_name, csv_file in csv_files.items():
    try:
        df = pd.read_csv(csv_file)
        df.to_sql(table_name, conn, if_exists='replace', index=False)
        print(f"Success: Table '{table_name}' loaded with {len(df)} rows.")
    except Exception as e:
        print(f"Error loading {csv_file}: {e}")


cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("\nTables found in database:", [t[0] for t in tables])

conn.close()

Loading tables into SQLite database...
Success: Table 'fund_master' loaded with 40 rows.
Success: Table 'aum_by_fund_house' loaded with 90 rows.
Success: Table 'monthly_sip_inflows' loaded with 48 rows.
Success: Table 'category_inflows' loaded with 144 rows.
Success: Table 'industry_folio_count' loaded with 21 rows.
Success: Table 'portfolio_holdings' loaded with 322 rows.
Success: Table 'benchmark_indices' loaded with 8050 rows.
Success: Table 'nav_history' loaded with 46000 rows.
Success: Table 'investor_transactions' loaded with 32778 rows.
Success: Table 'scheme_performance' loaded with 40 rows.

Tables found in database: ['fund_master', 'aum_by_fund_house', 'monthly_sip_inflows', 'category_inflows', 'industry_folio_count', 'portfolio_holdings', 'benchmark_indices', 'nav_history', 'investor_transactions', 'scheme_performance']


In [ ]:
import os
import sqlite3
import pandas as pd


try:
    from google.colab import files
except ImportError:
    files = None


csv_files = {
    'fund_master': 'cleaned_01_fund_master.csv',
    'aum_by_fund_house': 'cleaned_03_aum_by_fund_house.csv',
    'monthly_sip_inflows': 'cleaned_04_monthly_sip_inflows.csv',
    'category_inflows': 'cleaned_05_category_inflows.csv',
    'industry_folio_count': 'cleaned_06_industry_folio_count.csv',
    'portfolio_holdings': 'cleaned_09_portfolio_holdings.csv',
    'benchmark_indices': 'cleaned_10_benchmark_indices.csv',
    'nav_history': 'nav_history_cleaned.csv',
    'investor_transactions': 'investor_transactions_cleaned.csv',
    'scheme_performance': 'scheme_performance_cleaned.csv'
}

db_name = 'mutual_funds.db'


print(f"Connecting to database: {db_name}")
conn = sqlite3.connect(db_name)
cursor = conn.cursor()


print("\n--- Step 1: Importing Cleaned CSVs into SQLite ---")
loaded_count = 0
for table_name, csv_file in csv_files.items():
    if os.path.exists(csv_file):
        try:

            df = pd.read_csv(csv_file)

            df.to_sql(table_name, conn, if_exists='replace', index=False)
            print(f"Success: '{csv_file}' -> Loaded into table '{table_name}'")
            loaded_count += 1
        except Exception as e:
            print(f"Error loading '{csv_file}': {e}")
    else:
        print(f"Warning: File '{csv_file}' not found in the current folder path.")


print(f"\n--- Step 2: Verification ({loaded_count} Tables Found) ---")
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
sqlite_tables = [t[0] for t in cursor.fetchall()]

for table in sqlite_tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    row_count = cursor.fetchone()[0]
    cursor.execute(f"PRAGMA table_info({table})")
    columns = [col[1] for col in cursor.fetchall()]
    print(f"📋 Table: {table:<22} | Total Rows: {row_count:<6} | Key Columns: {columns[:3]}...")


conn.close()
print("\nDatabase compilation and verification successfully finished.")


if files:
    print("\n--- Step 3: Downloading 'mutual_funds.db' to your local computer ---")
    files.download(db_name)

Connecting to database: mutual_funds.db

--- Step 1: Importing Cleaned CSVs into SQLite ---
Success: 'cleaned_01_fund_master.csv' -> Loaded into table 'fund_master'
Success: 'cleaned_03_aum_by_fund_house.csv' -> Loaded into table 'aum_by_fund_house'
Success: 'cleaned_04_monthly_sip_inflows.csv' -> Loaded into table 'monthly_sip_inflows'
Success: 'cleaned_05_category_inflows.csv' -> Loaded into table 'category_inflows'
Success: 'cleaned_06_industry_folio_count.csv' -> Loaded into table 'industry_folio_count'
Success: 'cleaned_09_portfolio_holdings.csv' -> Loaded into table 'portfolio_holdings'
Success: 'cleaned_10_benchmark_indices.csv' -> Loaded into table 'benchmark_indices'
Success: 'nav_history_cleaned.csv' -> Loaded into table 'nav_history'
Success: 'investor_transactions_cleaned.csv' -> Loaded into table 'investor_transactions'
Success: 'scheme_performance_cleaned.csv' -> Loaded into table 'scheme_performance'

--- Step 2: Verification (10 Tables Found) ---
📋 Table: fund_master   

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>